Summarize a long text document into a concise and coherent abstract using LangChain's tools and pipelines.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: credentials from .env
# ============================================================================
# This repo keeps keys in a .env file at the project root (python-dotenv),
# rather than prompting with getpass -- so the notebook can be re-run without
# retyping a key, and nothing secret is ever typed into a saved output.
import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("✅ Environment loaded")

In [ ]:
# ============================================================================
# IMPORTS: splitter, prompt, model, parser
# ============================================================================
# Note what is NOT imported: `load_summarize_chain`. The original cell imported
# it and never used it. This notebook builds its map-reduce by hand instead.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
input_text = """ 
LangChain is a framework for building applications powered by large language models (LLMs). 
It simplifies the development of workflows by providing modular components for tasks like text generation, 
summarization, and knowledge retrieval. LangChain supports integration with external tools such as APIs 
and databases, allowing you to create dynamic and context-aware applications. Its key fea-tures include 
memory management for conversational agents, retrieval-augmented generation (RAG), and support for custom tools. You widely use LangChain for building chatbots, summarization tools, and knowledge-based 
applications. """

In [ ]:
# ============================================================================
# SPLIT: break the text into chunks
# ============================================================================
# RecursiveCharacterTextSplitter, not CharacterTextSplitter: the latter splits
# on blank lines ("\n\n") by default, and this input_text has none -- so it
# would return the whole text as ONE chunk and the map step below would have
# nothing to parallelize.
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_text(input_text)
print(f"📄 Split into {len(chunks)} chunk(s)")

In [ ]:
# Define the Prompt Template

prompt = PromptTemplate(
    input_variables= ["text"],
    template= "Summarize the following text: {text}")

In [ ]:
# ============================================================================
# MODEL: chat model, not the legacy completions endpoint
# ============================================================================
# Was: OpenAI(model="gpt-3.5-turbo-instruct") -- the legacy /completions API.
# The rest of this track uses chat models, and gpt-3.5-turbo-instruct is a
# retired completions model, so this notebook now matches its siblings.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
# llm = ChatGroq(model="openai/gpt-oss-120b")

print(f"🤖 Model loaded: {llm.model_name}")

In [ ]:
# ============================================================================
# MAP STEP: one LCEL chain, batched over every chunk
# ============================================================================
# Was: LLMChain(llm=llm, prompt=prompt) then a list comprehension of .run().
# LCEL replaces the class outright -- and `.batch()` runs the chunks
# concurrently, which the old per-chunk .run() loop did not.
summarization_chain = prompt | llm | StrOutputParser()

summaries = summarization_chain.batch([{"text": chunk} for chunk in chunks])

print(f"🔧 Summarized {len(summaries)} chunk(s)")

In [ ]:
final_summary = " ".join(summaries)
print(final_summary)

In [ ]:
# ============================================================================
# REDUCE STEP: combine the per-chunk summaries into one
# ============================================================================
# This is the actual reduce: it summarizes the *summaries* produced by the map
# step above, which is what makes the pair a map-reduce.
#
# Contrast it with the single-pass baseline -- summarizing the raw text in one
# call -- which is the "stuff" strategy and is what the original cell did:
#     summarization_chain.invoke({"text": input_text})
# Stuff is simpler and better when the text fits in one context window;
# map-reduce is what you reach for when it does not.
# `final_summary` is the joined map output, bound in the previous cell --
# reusing it makes the data flow between the two steps explicit.
summarization_chain.invoke({"text": final_summary})

![image.png](attachment:image.png)